In [5]:
import rasterio
import os
import datetime

import colorcet as cc
import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
import pandas as pd
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import HoverTool, Title
from pathlib import Path
import folium
import rioxarray
import netCDF4

from shapely.geometry import box

from blackmarble.extract import bm_extract
from blackmarble.raster import bm_raster

import xarray as xr
import numpy as np

#Local Functions
import sys
sys.path.append('../src')
from ntl_functions import plot_NASA_NTL, filter_dataset_by_shapefile

%load_ext autoreload
%autoreload 2

plt.rcParams["figure.figsize"] = (18, 10)
pd.set_option('display.max_rows', 50)

In [8]:
#Somalia Shape File Downlaoded from. 
#https://gadm.org/download_country.html
gdf = gpd.read_file(
    "gadm41_SOM_1.json.zip"
)
#Somaliland SHP File
somaliland_shp = gdf[gdf["HASC_1"].isin(["SO.AW", "SO.WO", "SO.TO", "SO.SA", "SO.SO"])]



In [31]:
file_path = 'Somaliland_Only.tif'

# Open the file
with rasterio.open(file_path) as src:
    raster_data = src.read()  # Read the first band
    profile = src.profile  # Metadata about the file


In [56]:
# List of years corresponding to each band
time = np.arange(2012, 2024)

# Open the raster file
with rasterio.open(file_path) as src:
    raster_data = src.read()  # Shape: (bands, height, width)
    
    # Get metadata for coordinates
    transform = src.transform
    crs = src.crs
    width = src.width
    height = src.height

    # Generate coordinate arrays
    lon = np.arange(0, width) * transform.a + transform.c
    lat = np.arange(0, height) * transform.e + transform.f

# Create the xarray Dataset with 'year' as a dimension
raster_dataset = xr.Dataset(
    data_vars={
        'raster_values': (['time', 'lat', 'lon'], raster_data)
    },
    coords={
        'time': time,
        'lat': lat,
        'lon': lon
    },
    attrs={
        'crs': crs.to_string(),
        'transform': transform
    }
)

# Display the Dataset
print(raster_dataset)

<xarray.Dataset> Size: 62MB
Dimensions:        (time: 12, lat: 843, lon: 1535)
Coordinates:
  * time           (time) int32 48B 2012 2013 2014 2015 ... 2020 2021 2022 2023
  * lat            (lat) float64 7kB 11.51 11.51 11.5 11.5 ... 8.012 8.008 8.004
  * lon            (lon) float64 12kB 42.68 42.69 42.69 ... 49.07 49.07 49.08
Data variables:
    raster_values  (time, lat, lon) float32 62MB nan nan nan nan ... nan nan nan
Attributes:
    crs:        EPSG:4326
    transform:  | 0.00, 0.00, 42.68|\n| 0.00,-0.00, 11.51|\n| 0.00, 0.00, 1.00|


In [58]:
raster_dataset

<xarray.Dataset> Size: 62MB
Dimensions:        (time: 12, lat: 843, lon: 1535)
Coordinates:
  * time           (time) int32 48B 2012 2013 2014 2015 ... 2020 2021 2022 2023
  * lat            (lat) float64 7kB 11.51 11.51 11.5 11.5 ... 8.012 8.008 8.004
  * lon            (lon) float64 12kB 42.68 42.69 42.69 ... 49.07 49.07 49.08
Data variables:
    raster_values  (time, lat, lon) float32 62MB nan nan nan nan ... nan nan nan
Attributes:
    crs:        EPSG:4326
    transform:  | 0.00, 0.00, 42.68|\n| 0.00,-0.00, 11.51|\n| 0.00, 0.00, 1.00|

In [60]:
## Do descriptive statistics on this data. 

In [ ]:
plot_NASA_NTL(log_Berbera_NTL, Berbera, date = "2012-01-01", 
              variable = "NearNadir_Composite_Snow_Free",
              title_prefix = "Berbera NTL",
              robust = False, vmin=0, vmax = log_Berbera_NTL["NearNadir_Composite_Snow_Free"].max())

In [36]:
SL_VNP46A4_EY_2012_24

<xarray.Dataset> Size: 125MB
Dimensions:                        (time: 12, y: 848, x: 1531)
Coordinates:
  * x                              (x) float64 12kB 42.71 42.71 ... 49.07 49.08
  * y                              (y) float64 7kB 11.51 11.51 ... 7.99 7.986
  * time                           (time) datetime64[ns] 96B 2012-01-01 ... 2...
Data variables:
    NearNadir_Composite_Snow_Free  (time, y, x) float64 125MB nan nan ... nan
Attributes: (12/39)
    AlgorithmType:                     b'SCI'
    AlgorithmVersion:                  b'NPP_PR46A3 2.0.0'
    AREA_OR_POINT:                     Area
    Conventions:                       b'CF-1.6'
    DataResolution:                    b'15 arc second'
    DayNightFlag:                      b'Night'
    ...                                ...
    NorthBoundingCoord:                10.0
    publisher_email:                   b'modis-ops@lists.nasa.gov'
    publisher_name:                    b'LAADS'
    publisher_url:                     b'https://ladsweb.modaps.eosdis.nasa.gov'
    SouthBoundingCoord:                0.0
    WestBoundingCoord:                 40.0

In [34]:
SL_VNP46A4_EY_2012_24 = xr.open_dataset("Combined Datasets/SL_VNP46A4_EY_2012_24.nc")
SL_VNP46A4_EY_2012_24 = SL_VNP46A4_EY_2012_24.where(SL_VNP46A4_EY_2012_24 != 6553.5, 0)

logged_NTL = xr.where(
    SL_VNP46A4_EY_2012_24 > 0, 
    np.log(SL_VNP46A4_EY_2012_24), 
    SL_VNP46A4_EY_2012_24  # Assign 0 for non-positive values
)

C:\Users\samsa\anaconda3\Lib\site-packages\xarray\core\computation.py:824: RuntimeWarning: divide by zero encountered in log
  result_data = func(*input_data)
